# Stage 1 — Dataset Generation for RAG

## Project Overview

This notebook implements the first stage of the project: the automatic generation of an instruction-response dataset from a domain-specific document.

The knowledge source used in this work is the official user manual of the **Midea MFM01D110WB 11kg Washing Machine**. The generated dataset will be used in subsequent stages for Retrieval-Augmented Generation (RAG), LoRA-based fine-tuning, model evaluation, and RESTful API deployment.

---

## Objectives

- Extract textual content from the PDF manual.
- Split the document into manageable chunks.
- Generate instruction-response pairs using a local Large Language Model (LLM).
- Validate and curate the generated examples.
- Export the final dataset in JSONL format for fine-tuning.

## Knowledge Source

**Document:** Midea MFM01D110WB 11kg Washing Machine User Manual

The manual contains operational instructions, installation guidelines, safety recommendations, maintenance procedures, troubleshooting information, and technical specifications. These characteristics make it an appropriate domain-specific knowledge source for instruction tuning and question-answering tasks.

## Language Model

The instruction-response pairs are generated using:

- **Model:** Qwen/Qwen2.5-1.5B-Instruct
- **Parameters:** 1.5 Billion
- **Task:** Instruction-response generation

## Expected Output

The final dataset must follow the JSONL format below:

```json
{
  "Instruction": "Example question",
  "Output": "Example answer"
}
```

The resulting dataset will serve as the training corpus for the LoRA fine-tuning stage.

---


## 1. Installing Dependencies

This section installs the libraries required for document processing, dataset generation, language model inference, and data manipulation.

The dependencies include:

- **pdfplumber** for PDF text extraction.
- **transformers** for loading and running Large Language Models (LLMs).
- **torch** as the deep learning framework.
- **accelerate** for optimized model execution.
- **tqdm** for progress monitoring during dataset generation.

These libraries provide the foundation for the Retrieval-Augmented Generation (RAG) dataset creation pipeline implemented in this notebook.


In [1]:
%pip install transformers torch accelerate pdfplumber tqdm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Importing Libraries

This section imports the Python libraries required for PDF processing, dataset generation, model inference, and file manipulation.


In [2]:
import json
import re

import pdfplumber
import torch

from tqdm import tqdm
from transformers import pipeline, logging

# Display only critical Transformers messages
logging.set_verbosity_error()

c:\Users\analu\Documents\Ana Luiza\Documentos - TAIAA\midea-mfm01d110wb-rag-lora-api\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. Extracting Text from the Knowledge Source

This section defines the function responsible for extracting textual content from the PDF document that serves as the project's knowledge base.


In [3]:
def extract_text_from_file(file_path):
    """
    Extracts text from a PDF or TXT file.

    Parameters
    ----------
    file_path : str
        Path to the input file.

    Returns
    -------
    str
        Complete extracted text content.
    """

    if file_path.lower().endswith(".pdf"):
        text = ""

        with pdfplumber.open(file_path) as pdf:
            for page in pdf.pages:
                page_text = page.extract_text()

                if page_text:
                    text += page_text + "\n"

        return text

    elif file_path.lower().endswith(".txt"):

        with open(file_path, "r", encoding="utf-8") as f:
            return f.read()

    else:
        raise ValueError(
            "Unsupported file format. Please use PDF or TXT files."
        )
    
PDF_PATH = "../data/pdf/manual.pdf"

manual_text = extract_text_from_file(PDF_PATH)

print(f"Total characters extracted: {len(manual_text):,}")

print("\nText preview:\n")
print(manual_text[:1000])

Cannot set stroke color: 2 components specified, but only 1 (grayscale), 3 (RGB), and 4 (CMYK) are supported
Cannot set non-stroke color: 2 components specified, but only 1 (grayscale), 3 (RGB), and 4 (CMYK) are supported


Total characters extracted: 81,898

Text preview:

LAVA E SECA
MANUAL DO USUÁRIO
11kg
MODELOS:
MFM01D110WB
www.midea.com/br
Obrigado por escolher a Midea!
A Midea é uma empresa comprometida com o bem-estar das pessoas. Com a
combinação de design inteligente e tecnologia, seu novo equipamento trará
novas experiências e deixará seu dia a dia muito mais agradável. Uma receita
simples que fez da Midea uma das maiores fabricantes de eletrodomésticos e
condicionadores de ar do mundo.
Este manual foi feito especialmente para que você conheça todas as características
do seu aparelho, além de informações sobre manutenção, execução de serviços
e claro, como obter o máximo das suas funcionalidades.
Caso precise de informações adicionais ou tenha dúvidas sobre a garantia,
entre em contato através do nosso Serviço de Atendimento ao Consumidor,
pelos telefones ou pelo site abaixo.
SAC - Serviço de Atendimento ao Consumidor
3003 1005 (capitais e regiões metropolitanas)
0800 648 1005 (demais localidad

- ### Saving Extracted Text


In [12]:
with open(
    "../data/processed/manual_extracted.txt",
    "w",
    encoding="utf-8"
) as f:
    f.write(manual_text)

print("Extracted text saved successfully.")

Extracted text saved successfully.


## 4. Text Chunking Strategy

Large Language Models cannot efficiently process an entire document at once due to context length limitations. Therefore, the extracted text must be divided into smaller segments, commonly referred to as chunks.

The chunking process directly impacts the quality of the generated instruction-response pairs. Smaller chunks tend to produce more focused questions and answers, while larger chunks provide additional context but may introduce irrelevant information.

---

To evaluate this trade-off, two chunk sizes will be tested:

- 500 characters
- 1000 characters

The resulting datasets will be compared based on the number and quality of generated instruction-response pairs.


In [4]:
def split_text(text, max_chunk_length=500):
    """
    Splits the document into chunks based on line breaks.

    Parameters
    ----------
    text : str
        Input text extracted from the document.

    max_chunk_length : int
        Maximum number of characters per chunk.

    Returns
    -------
    list
        List of text chunks.
    """

    paragraphs = text.split("\n")

    chunks = []
    current_chunk = ""

    for paragraph in paragraphs:

        if len(current_chunk) + len(paragraph) < max_chunk_length:
            current_chunk += paragraph + "\n"

        else:
            if current_chunk:
                chunks.append(current_chunk.strip())

            current_chunk = paragraph + "\n"

    if current_chunk:
        chunks.append(current_chunk.strip())

    return chunks

- ### Testing Chunk Size: 500


In [5]:
chunks_500 = split_text(
    manual_text,
    max_chunk_length=500
)

print(f"Number of chunks (500 chars): {len(chunks_500)}")

Number of chunks (500 chars): 176


- ### Chunk Size Example:


In [6]:
print(chunks_500[0])

LAVA E SECA
MANUAL DO USUÁRIO
11kg
MODELOS:
MFM01D110WB
www.midea.com/br
Obrigado por escolher a Midea!
A Midea é uma empresa comprometida com o bem-estar das pessoas. Com a
combinação de design inteligente e tecnologia, seu novo equipamento trará
novas experiências e deixará seu dia a dia muito mais agradável. Uma receita
simples que fez da Midea uma das maiores fabricantes de eletrodomésticos e
condicionadores de ar do mundo.


- ### Testing Chunk Size: 1000


In [7]:
chunks_1000 = split_text(
    manual_text,
    max_chunk_length=1000
)

print(f"Number of chunks (1000 chars): {len(chunks_1000)}")

Number of chunks (1000 chars): 85


- ### Chunk Size Example:


In [8]:
print(chunks_1000[0])

LAVA E SECA
MANUAL DO USUÁRIO
11kg
MODELOS:
MFM01D110WB
www.midea.com/br
Obrigado por escolher a Midea!
A Midea é uma empresa comprometida com o bem-estar das pessoas. Com a
combinação de design inteligente e tecnologia, seu novo equipamento trará
novas experiências e deixará seu dia a dia muito mais agradável. Uma receita
simples que fez da Midea uma das maiores fabricantes de eletrodomésticos e
condicionadores de ar do mundo.
Este manual foi feito especialmente para que você conheça todas as características
do seu aparelho, além de informações sobre manutenção, execução de serviços
e claro, como obter o máximo das suas funcionalidades.
Caso precise de informações adicionais ou tenha dúvidas sobre a garantia,
entre em contato através do nosso Serviço de Atendimento ao Consumidor,
pelos telefones ou pelo site abaixo.
SAC - Serviço de Atendimento ao Consumidor
3003 1005 (capitais e regiões metropolitanas)
0800 648 1005 (demais localidades)
https://www.midea.com/br/contato/


- ### Comparison:


In [9]:
print(f"500-char chunks:  {len(chunks_500)}")
print(f"1000-char chunks: {len(chunks_1000)}")

500-char chunks:  176
1000-char chunks: 85


## 5. Loading the Language Model

This section loads the Large Language Model (LLM) responsible for generating instruction-response pairs from the document chunks.

The selected model is **Qwen2.5-1.5B-Instruct**, an instruction-tuned language model with approximately 1.5 billion parameters. This model satisfies the project requirement of using a local LLM with more than 1.5B parameters.

The model will be used to generate question-answer pairs based exclusively on the content extracted from the Midea MFM01D110WB washing machine user manual.


In [10]:
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

print(f"Loading model: {MODEL_ID}")

Loading model: Qwen/Qwen2.5-1.5B-Instruct


In [11]:
hf_pipeline = pipeline(
    task="text-generation",
    model=MODEL_ID,
    device_map="cpu"
)

print("Model loaded successfully.")

Loading weights: 100%|██████████| 338/338 [00:00<00:00, 1544.62it/s]


Model loaded successfully.


## 6. Generating Instruction-Response Pairs

This section generates instruction-response pairs from the document chunks using the selected language model.

Each generated pair consists of:

- An instruction (question).
- A response (answer).

Both elements must be derived exclusively from the information contained in the source chunk.


### 6.1 Chunk Quality Filter

Some chunks may contain metadata such as website URLs, phone numbers, copyright notices, or table of contents entries. These chunks are not useful for instruction tuning and may generate low-quality instruction-response pairs.

To improve dataset quality, a filtering function is applied before generation.


In [12]:
def is_relevant_chunk(chunk):
    """
    Filters chunks that are unlikely to generate useful
    instruction-response pairs.
    """

    chunk_lower = chunk.lower()

    unwanted_patterns = [
        "www.",
        "http",
        "0800",
        "sac",
        "atendimento ao consumidor",
        "sumário",
        "índice",
        "descrição do aparelho",
        "características técnicas gerais",
        "conectividade wifi",
        "instalando o aplicativo",
        "configurando o aplicativo"
    ]

    if any(pattern in chunk_lower for pattern in unwanted_patterns):
        return False

    if len(chunk.split()) < 20:
        return False

    return True

### 6.2 Instruction-Response Generation

The following function prompts the language model to generate a practical user question and its corresponding answer based exclusively on the content contained in a document chunk.

The prompt is designed to prioritize operational, maintenance, safety, and troubleshooting information related to the washing machine.


In [30]:
def generate_instruction_response(chunk, hf_pipeline):

    prompt = f"""
You are creating a supervised fine-tuning dataset from a washing machine user manual.

Generate ONE question-answer pair.

Requirements:

- The question must focus on a specific fact, warning, instruction, or recommendation found in the text.
- Avoid generic questions.
- Use only information explicitly present in the text.
- Do not infer information.
- Do not add explanations.
- The answer must be concise (maximum 15 words).
- Prefer safety, installation, operation, maintenance, and troubleshooting information.

Output format:

INSTRUCTION: <question>

RESPONSE: <answer>

TEXT:
{chunk}
"""

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    try:

        outputs = hf_pipeline(
            messages,
            max_new_tokens=100,
            return_full_text=False,
            do_sample=False
        )

        generated_text = outputs[0]["generated_text"]

        if "INSTRUCTION:" not in generated_text:
            return None, None

        if "RESPONSE:" not in generated_text:
            return None, None

        instruction = (
            generated_text
            .split("INSTRUCTION:")[1]
            .split("RESPONSE:")[0]
            .strip()
        )

        response = (
            generated_text
            .split("RESPONSE:")[1]
            .strip()
        )

        return instruction, response

    except Exception as e:

        print(f"Generation error: {e}")

        return None, None

### 6.3 Single Example Test

Before generating the full dataset, a single chunk is tested to verify that the model produces instruction-response pairs in the expected format.


In [39]:
test_chunk = chunks_500[19]

print(test_chunk)

1.1 - Medidas Importantes de
Segurança
NOTA
Para reduzir os riscos de choques elétricos,
• Este aparelho não deve ser instalado
queimaduras, ferimentos pessoais ou danos ao
embutido.
equipamento, siga as recomendações básicas
• Evite instalar o aparelho em um local
de segurança ao usar este aparelho:
onde exista a incidência de raios solares.
• O aparelho não deve ser instalado atrás
PERIGO
de uma porta com fechadura, uma


In [40]:
instruction, response = generate_instruction_response(
    chunks_500[19],
    hf_pipeline
)

print("Instruction:")
print(instruction)

print()

print("Response:")
print(response)

Instruction:
What should you avoid when installing the appliance?

Response:
Avoid installing it behind a door with a handle, as this can cause electrical shocks.


### 6.4 Multiple Example Test

A small sample of chunks is evaluated to assess the quality, consistency, and relevance of the generated instruction-response pairs.


In [41]:
for i in range(18, 25):

    instruction, response = generate_instruction_response(
        chunks_500[i],
        hf_pipeline
    )

    print("=" * 80)
    print(f"CHUNK {i}")

    print()
    print("Instruction:")
    print(instruction)

    print()
    print("Response:")
    print(response)

    print("\n")

CHUNK 18

Instruction:
What should you check before using the appliance?

Response:
Ensure that all necessary electrical connections, such as a water supply line, plumbing work, and electrical wiring, have been completed correctly. Always consult with an electrician if unsure about any aspect of the setup.


CHUNK 19

Instruction:
What should you avoid when installing the appliance?

Response:
Avoid installing it behind a door with a handle, as this can cause electrical shocks.


CHUNK 20

Instruction:
What is the caution regarding the use of this appliance?

Response:
People with reduced mental capacity should not operate this appliance without proper instructions.


CHUNK 21

Instruction:
What should you avoid doing while using the washer?

Response:
Never place your hands under the appliance.


CHUNK 22

Instruction:
What is important to avoid when installing the washing machine?

Response:
Ferments can occur if installed improperly.


CHUNK 23

Instruction:
What should you check if

## 7. Full Dataset Generation

After validating the chunking strategy and the instruction-response generation process, the complete dataset is generated using all relevant chunks extracted from the Midea MFM01D110WB washing machine user manual.

Each relevant chunk is processed by the language model to generate one instruction-response pair. The resulting examples are stored in JSONL format and will later be manually curated to remove invalid or hallucinated samples.

### 7.1 Save Dataset Function

In [ ]:
def save_to_jsonl(pairs, output_file):
    """
    Saves instruction-response pairs to a JSONL file.
    """

    with open(output_file, "w", encoding="utf-8") as f:

        for instruction, response in pairs:

            example = {
                "Instruction": instruction,
                "Output": response
            }

            f.write(
                json.dumps(
                    example,
                    ensure_ascii=False
                ) + "\n"
            )

### 7.2 Generate Dataset

The following step processes all relevant chunks and generates instruction-response pairs using the selected language model.

In [ ]:
pairs = []

total_chunks = len(chunks_500)
relevant_chunks = 0
failed_generations = 0

for chunk in tqdm(
    chunks_500,
    desc="Generating instruction-response pairs"
):

    if not is_relevant_chunk(chunk):
        continue

    relevant_chunks += 1

    instruction, response = generate_instruction_response(
        chunk,
        hf_pipeline
    )

    if instruction and response:

        pairs.append(
            (
                instruction.strip(),
                response.strip()
            )
        )

    else:

        failed_generations += 1

### 7.3 Dataset Statistics

The following statistics summarize the dataset generation process.

In [ ]:
print("=" * 60)

print(f"Total chunks: {total_chunks}")
print(f"Relevant chunks: {relevant_chunks}")
print(f"Generated pairs: {len(pairs)}")
print(f"Failed generations: {failed_generations}")

print("=" * 60)

### 7.4 Remove Duplicate Examples

Duplicate instruction-response pairs are removed before saving the final dataset.

In [ ]:
unique_pairs = list(set(pairs))

duplicates_removed = len(pairs) - len(unique_pairs)

print(f"Duplicates removed: {duplicates_removed}")
print(f"Unique pairs: {len(unique_pairs)}")

### 7.5 Save Dataset

The final dataset is saved in JSONL format for the fine-tuning stage.

In [ ]:
OUTPUT_FILE = "../data/processed/dataset_gerado.jsonl"

save_to_jsonl(
    unique_pairs,
    OUTPUT_FILE
)

print(f"Dataset saved successfully: {OUTPUT_FILE}")

### 7.6 Dataset Preview

A random sample of generated examples is displayed for inspection.

In [ ]:
import random

sample_size = min(10, len(unique_pairs))

for instruction, response in random.sample(
    unique_pairs,
    sample_size
):

    print("-" * 80)

    print("Instruction:")
    print(instruction)

    print()

    print("Response:")
    print(response)

    print()